# LLM Integration Testing

This notebook verifies Gemini LLM connectivity, basic conversational support, and health summary generation based on predictions, SHAP, and recommendations.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Append parent directory to path so we can import our modules
sys.path.append(os.path.abspath(os.path.join("..")))

# Load environment variables from .env file
load_dotenv(dotenv_path=os.path.join("..", ".env"))

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    print("Warning: GEMINI_API_KEY not found in environment or .env file.")
    print("Please create a .env file with GEMINI_API_KEY=your_key in the project root.")
else:
    print("Gemini API key is configured.")

## 1. Test Conversational Chatbot

In [ ]:
from llm.chatbot import MaatriChatbot

chatbot = MaatriChatbot(api_key=api_key)

query = "What foods help with anemia during pregnancy?"
print(f"User: {query}\n")

try:
    response = chatbot.generate_response(query)
    print(f"Maatri AI:\n{response}")
except Exception as e:
    print(f"Error generating response: {e}")

## 2. Test AI Health Summary Generator

In [ ]:
from llm.summary_generator import MaatriSummaryGenerator
from ml.risk_prediction.predict import predict_risk
from ml.explainability.shap_explainer import MaatriSHAPExplainer
from ml.recommendation_engine.recommendations import get_recommendations

# Sample clinical inputs (High risk case)
patient_data = {
    'Age': 35,
    'SystolicBP': 140,
    'DiastolicBP': 95,
    'BS': 8.2,
    'BodyTemp': 98.6,
    'HeartRate': 85
}

# 1. Run prediction
pred_result = predict_risk(patient_data, models_dir=os.path.join("..", "models"))

# 2. Run SHAP explanation
risk_mapping = {'Low Risk': 0, 'Mid Risk': 1, 'High Risk': 2}
pred_idx = risk_mapping[pred_result['risk_category']]
explainer = MaatriSHAPExplainer(models_dir=os.path.join("..", "models"))
shap_explanation = explainer.explain_instance(patient_data, pred_idx)

# 3. Run Recommendations
recs_result = get_recommendations(pred_result['risk_category'], patient_data)

# 4. Generate summary using Gemini
summary_gen = MaatriSummaryGenerator(api_key=api_key)
try:
    summary = summary_gen.generate_health_summary(pred_result, shap_explanation, recs_result)
    print("=== Generated Patient Health Summary ===")
    print(summary)
except Exception as e:
    print(f"Error generating summary: {e}")